In [ ]:
import numpy as np 
import xarray as xr
import matplotlib.pyplot as plt 
import mixsea as mx


In [ ]:
# load bathymetry data 

bath = xr.open_dataset('data/BedMachine_TNB_subset.nc')


# load mooring data 
# LOAD IN SAVED INTERMEDIATE FILES - MAC 

TNBD_ctd_top = xr.open_dataset('data/TNBD_ctd_top_30min.nc')
TNBD_ctd_bot = xr.open_dataset('data/TNBD_ctd_bot_30min.nc')
TNBD_vel_top = xr.open_dataset('data/TNBD_vel_top_30min.nc')
TNBD_vel_bot = xr.open_dataset('data/TNBD_vel_bot_30min.nc')

DITD_ctd_top = xr.open_dataset('data/DITD_ctd_top_30min.nc')
DITD_ctd_bot = xr.open_dataset('data/DITD_ctd_bot_30min.nc')
DITD_vel_top = xr.open_dataset('data/DITD_vel_top_30min.nc')
DITD_vel_bot = xr.open_dataset('data/DITD_vel_bot_30min.nc')      # sensor cut out in june 2018

D_ctd_top = xr.open_dataset('data/D_ctd_top_30min.nc')
D_ctd_mid = xr.open_dataset('data/D_ctd_mid_30min.nc')
# D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
D_vel_mid = xr.open_dataset('data/D_vel_mid_30min.nc')
D_vel_bot = xr.open_dataset('data/D_vel_bot_30min.nc')

In [ ]:
mx.nsq.adiabatic_leveling?

In [ ]:
# load in CTD data
pipers = xr.open_dataset('data/pipers_ctd_58stations.nc')
palmer = xr.open_dataset('data/Palmer_CTD_41stations.nc')

In [ ]:
pipers

In [ ]:
import numpy as np
import xarray as xr
import gsw
import mixsea as mx

def add_n2_and_rd(ds, sal_var, temp_var, press_var, depth_var, bin_width):
    """
    Compute Brunt-Vaisala frequency squared (N2, via adiabatic leveling)
    and Rossby radius of deformation (Rd) for each station in ds,
    and add them back into ds as new variables.
    """
    stations = ds['station'].values

    n2_out = np.full(ds[sal_var].shape, np.nan)
    rd_out = np.full(len(stations), np.nan)

    for i, st in enumerate(stations):
        cast = ds.sel(station=st)

        S = cast[sal_var].values
        T = cast[temp_var].values
        p = cast[press_var].values
        lat = float(cast['lat'])
        lon = float(cast['lon'])

        valid = np.isfinite(S) & np.isfinite(T) & np.isfinite(p)
        if valid.sum() < 2:
            continue

        N2 = mx.nsq.adiabatic_leveling(
            P=p[valid],
            S=S[valid],
            T=T[valid],
            lon=lon,
            lat=lat,
            bin_width=bin_width,
            cap=None
        )

        n2_out[i, valid] = N2

        N = np.where(N2 > 0, np.sqrt(N2), np.nan)
        H = float(cast[depth_var].max())
        f = np.abs(gsw.f(lat))

        rd_out[i] = (np.nanmean(N) * H) / f

    ds = ds.assign(
        N2=(('station', 'sample'), n2_out),
        Rd=(('station',), rd_out)
    )

    ds['N2'].attrs = {
        'long_name': 'Brunt-Vaisala frequency squared',
        'units': 's^-2',
        'method': 'mixsea adiabatic leveling'
    }
    ds['Rd'].attrs = {
        'long_name': 'Rossby radius of deformation',
        'units': 'm',
        'method': 'mean(N) * H / f'
    }

    return ds


palmer = add_n2_and_rd(palmer, sal_var='Salinity', temp_var='Temperature',
                        press_var='Pressure', depth_var='Depth',bin_width=100)

pipers = add_n2_and_rd(pipers, sal_var='sal', temp_var='temp',
                        press_var='press', depth_var='depth',bin_width=100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Font sizes tuned for a 6.5in-wide figure inserted at \textwidth
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

# Convert to km for readability
palmer_rd = palmer['Rd'].values / 1000
pipers_rd = pipers['Rd'].values / 1000

palmer_rd = palmer_rd[np.isfinite(palmer_rd)]
pipers_rd = pipers_rd[np.isfinite(pipers_rd)]

# Shared bin edges so both histograms use identical bin widths
all_rd = np.concatenate([palmer_rd, pipers_rd])
bins = np.linspace(all_rd.min(), all_rd.max(), 16)  # 15 bins across the combined range

fig, ax = plt.subplots(figsize=(6.5, 4.5))

ax.hist(palmer_rd, bins=bins, density=True, alpha=0.5, color='darkorange', label='Palmer', edgecolor='k', linewidth=0.4)
ax.hist(pipers_rd, bins=bins, density=True, alpha=0.5, color='mediumpurple', label='Pipers', edgecolor='k', linewidth=0.4)

ax.set_xlabel('Rossby Radius of Deformation, Rd (km)')
# ax.set_ylabel('Probability Density')
ax.set_ylabel('Probability Density (km$^{-1}$)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rd_pdf_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### estimating vertical extent of HSSW layer 


In [ ]:
# load in CTD data 
palmer = xr.open_dataset('data/Palmer_ctd_DrygalskiBasin.nc')
pipers = xr.open_dataset('data/PIPERS_ctd_DrygalskiBasin.nc')

In [ ]:
palmer = add_n2_and_rd(palmer, sal_var='Salinity', temp_var='Temperature',
                        press_var='Pressure', depth_var='Depth',bin_width=100)

pipers = add_n2_and_rd(pipers, sal_var='sal', temp_var='temp',
                        press_var='press', depth_var='depth',bin_width=100)

In [ ]:
def add_dist_from_bottom(ds, depth_var="depth"):
    """
    Add a 'dist_from_bottom' variable to the dataset: distance above
    the seafloor for each sample, computed per-station.

    dist_from_bottom = bottom_depth(station) - depth(station, sample)

    So dist_from_bottom = 0 at the deepest sample of each cast,
    and increases toward the surface.
    """
    depth = ds[depth_var]

    # bottom depth per station = max valid depth in that profile
    bottom_depth = depth.max(dim="sample", skipna=True)

    dist_from_bottom = bottom_depth - depth
    dist_from_bottom.attrs["long_name"] = "Distance above bottom"
    dist_from_bottom.attrs["units"] = "m"

    ds = ds.assign(dist_from_bottom=dist_from_bottom)
    return ds

In [ ]:
pipers = add_dist_from_bottom(pipers)

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).Absolute_Salinity,pipers.isel(station=st).dist_from_bottom,color='k')
# plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.9, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Absolute Salinity [g/kg]')
plt.title('PIPERS CTD Data')

In [ ]:
pipers.time.values

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).Absolute_Salinity,pipers.isel(station=st).depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.9, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Absolute Salinity [g/kg]')
plt.title('PIPERS CTD Data')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Font sizes tuned for Overleaf (6.5in text width) — widen slightly since this is 4 panels
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 8,
    'legend.fontsize': 6.5,
})

n_stations = len(pipers.station)
# colors = cm.viridis(np.linspace(0, 1, n_stations))

fig, axes = plt.subplots(1, 4, figsize=(9, 5), sharey=True)

variables = ['Absolute_Salinity', 'Potential_Temperature', 'Potential_Density', 'N2']
xlabels = [
    'Absolute Salinity [g/kg]',
    'Potential Temperature [°C]',
    'Potential Density [kg m$^{-3}$]',
    'N$^2$ [s$^{-2}$]'
]

for i, (var, xlabel) in enumerate(zip(variables, xlabels)):
    ax = axes[i]
    for st in range(n_stations):
        cast = pipers.isel(station=st)
        ax.plot(cast[var], cast.depth, color='mediumpurple', lw=0.8, alpha=0.8)
    ax.set_xlabel(xlabel)
    ax.grid(alpha=0.3)

axes[0].axvline(x=34.9, color='k', linestyle='--', linewidth=1)
axes[0].set_ylabel('Depth [m]')
axes[0].invert_yaxis()  # shared y-axis, so this flips all panels

# fig.suptitle('PIPERS CTD Data', y=1.02)

# Colorbar as a stand-in "legend" for station number (since 58 individual labels would be unreadable)
# sm = cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=pipers.station.min(), vmax=pipers.station.max()))
# sm.set_array([])
# cbar = fig.colorbar(sm, ax=axes, orientation='vertical', fraction=0.03, pad=0.02)
# cbar.set_label('Station #', fontsize=8)
# cbar.ax.tick_params(labelsize=7)

# plt.savefig('pipers_ctd_4panel.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter
from pyproj import Transformer
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath

# Font sizes tuned for a 6.5in-wide figure inserted at \textwidth in Overleaf
plt.rcParams.update({
    'font.size': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'legend.title_fontsize': 7.5,
})

# ── USER INPUTS ───────────────────────────────────────────────────────────────
MOORINGS = {
    'DITD': {
        'lon': float(DITD_ctd_top.lon.values),
        'lat': float(DITD_ctd_top.lat.values),
    },
    'D': {
        'lon': float(D_vel_bot.lon.values),
        'lat': float(D_vel_bot.lat.values),
    },
    'TNBD': {
        'lon': float(TNBD_vel_bot.lon.values),
        'lat': float(TNBD_vel_bot.lat.values),
    },
}

MOORING_COLOR = 'steelblue'

# Trough-region bounds (original tight bounds)
LON_BOUNDS     = (161.9, 169.0)
LAT_BOUNDS     = (-75.8, -74.3)
DEPTH_CONTOURS = [200, 500, 800, 900, 1000, 1100, 1200]

# Geographic labels
GEO_LABELS = [
    {'text': 'Drygalski\nIce Tongue', 'lon': 163.2,  'lat': -75.35,
     'size': 7,  'style': 'italic', 'color': 'white'},
    {'text': 'Nansen\nIce Shelf',     'lon': 163,  'lat': -74.8,
     'size': 7,  'style': 'italic', 'color': 'white'},
    {'text': 'Drygalski\nTrough',     'lon': 166,  'lat': -74.95,
     'size': 7,  'style': 'italic', 'color': 'white'},
    {'text': 'Ross Sea',              'lon': 168,  'lat': -75.2,
     'size': 8,  'style': 'italic', 'color': 'white'},
]

# ── LOAD BEDMACHINE ───────────────────────────────────────────────────────────
transformer_inv = Transformer.from_crs('EPSG:3031', 'EPSG:4326',
                                        always_xy=True)
x_2d, y_2d     = np.meshgrid(bath['x'].values, bath['y'].values)
lon_2d, lat_2d = transformer_inv.transform(x_2d, y_2d)

bed  = bath['bed'].values.astype(float)
mask = bath['mask'].values

depth_raw  = np.where(bed < 0, -bed, np.nan)
ocean_mask = (mask == 0) | (mask == 3)
depth_plot = np.where(ocean_mask, depth_raw, np.nan)
depth_sm   = gaussian_filter(np.where(ocean_mask, depth_raw, 0), sigma=2)
depth_sm   = np.where(ocean_mask, depth_sm, np.nan)

in_bounds      = ((lon_2d >= LON_BOUNDS[0]) & (lon_2d <= LON_BOUNDS[1]) &
                   (lat_2d >= LAT_BOUNDS[0]) & (lat_2d <= LAT_BOUNDS[1]))
depth_inbounds = depth_plot[in_bounds & np.isfinite(depth_plot)]
vmax           = np.nanpercentile(depth_inbounds, 98) \
                 if depth_inbounds.size > 0 else 2000

# ── FIGURE ────────────────────────────────────────────────────────────────────
# Sized to match Overleaf's 6.5in text width (1in margins, letter paper)
fig, ax = plt.subplots(figsize=(6.5, 6))
fig.patch.set_facecolor('white')

# ── Bathymetry ────────────────────────────────────────────────────────────────
pcm = ax.pcolormesh(lon_2d, lat_2d, depth_plot,
                     cmap='Blues', shading='auto',
                     vmin=0, vmax=vmax, zorder=1)
cbar = plt.colorbar(pcm, ax=ax, pad=0.02, fraction=0.025, aspect=30)
cbar.set_label('Depth (m)', fontsize=8, labelpad=6)
cbar.ax.tick_params(labelsize=7)

# ── Land / grounded ice ───────────────────────────────────────────────────────
land = np.where((mask == 1) | (mask == 2), 1.0, np.nan)
ax.pcolormesh(lon_2d, lat_2d, land,
              cmap='Greys', shading='auto',
              vmin=0, vmax=1, alpha=0.85, zorder=2)

# ── Floating ice — translucent white + tab:red outline ───────────────────────
floating_white = np.where(mask == 3, 1.0, np.nan)
ax.pcolormesh(lon_2d, lat_2d, floating_white,
              cmap='Greys_r', shading='auto',
              vmin=0, vmax=1, alpha=0.3, zorder=3)

ax.contour(lon_2d, lat_2d, (mask == 3).astype(float),
           levels=[0.5],
           colors=['tab:red'],
           linewidths=1.2,
           linestyles='-',
           zorder=5)

# ── Isobath contours ──────────────────────────────────────────────────────────
valid_contours = [c for c in DEPTH_CONTOURS
                  if np.nanmin(depth_sm) < c < np.nanmax(depth_sm)]
if valid_contours:
    cs = ax.contour(lon_2d, lat_2d, depth_sm,
                     levels=valid_contours,
                     colors='white', linewidths=0.5,
                     alpha=0.5, zorder=6)
    ax.clabel(cs, levels=valid_contours[::2], fmt='%d m',
              fontsize=6, colors='white', inline=True,
              inline_spacing=3)

# ── Geographic labels ─────────────────────────────────────────────────────────
for lbl in GEO_LABELS:
    t = ax.text(lbl['lon'], lbl['lat'], lbl['text'],
                ha='center', va='center',
                fontsize=lbl['size'], style=lbl['style'],
                color=lbl['color'], zorder=7, alpha=0.95)
    t.set_path_effects([
        pe.withStroke(linewidth=1.8, foreground='#222233')])


# ── Mooring markers (small, single color, no labels) ──────────────────────────
mooring_lons = [m['lon'] for m in MOORINGS.values()]
mooring_lats = [m['lat'] for m in MOORINGS.values()]
ax.scatter(
    mooring_lons, mooring_lats,
    marker='o',
    s=30,
    color=MOORING_COLOR,
    edgecolor='white',
    linewidth=0.8,
    zorder=9,
    label='Mooring'
)

# ── Pipers CTD stations (main feature — large markers) ────────────────────────
ax.scatter(
    pipers['lon'].values, pipers['lat'].values,
    marker='o',
    s=60,
    color='mediumpurple',
    edgecolor='k',
    linewidth=0.6,
    zorder=8,
    label='PIPERS CTD'
)

# ── North arrow ───────────────────────────────────────────────────────────────
ax.annotate('', xy=(0.96, 0.20), xytext=(0.96, 0.13),
            xycoords='axes fraction',
            arrowprops=dict(arrowstyle='->', color='white',
                            lw=1.8, mutation_scale=12))
ax.text(0.96, 0.10, 'N', transform=ax.transAxes,
        ha='center', va='top', fontsize=9,
        color='white', fontweight='bold')

# ── Legend ────────────────────────────────────────────────────────────────────
ice_patch = mpatches.Patch(facecolor='white', edgecolor='tab:red',
                            linewidth=1.0, label='Floating ice')
ctd_handle = plt.Line2D([0], [0], marker='s', color='none',
                        markerfacecolor='mediumpurple',
                        markeredgecolor='k',
                        markeredgewidth=0.6,
                        markersize=9, label='PIPERS CTD')
mooring_handle = plt.Line2D([0], [0], marker='o', color='none',
                            markerfacecolor=MOORING_COLOR,
                            markeredgecolor='white',
                            markeredgewidth=0.8,
                            markersize=6, label='Mooring')

leg = ax.legend(handles=[ctd_handle, mooring_handle, ice_patch],
                loc='upper left', fontsize=7,
                framealpha=0.75, edgecolor='#aaaaaa',
                facecolor='#1a2a3a', labelcolor='white',
                title='Legend', title_fontsize=7.5)
leg.get_title().set_color('white')

# ── Axes formatting ───────────────────────────────────────────────────────────
lat_mean = np.mean(LAT_BOUNDS)
ax.set_aspect(1.0 / np.cos(np.deg2rad(lat_mean)))
ax.set_xlim(LON_BOUNDS)
ax.set_ylim(LAT_BOUNDS)
ax.set_xlabel('Longitude (°E)', fontsize=9, labelpad=5)
ax.set_ylabel('Latitude (°N)',  fontsize=9, labelpad=5)
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:.1f}°E'))
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda y, _: f'{y:.1f}°'))
ax.tick_params(labelsize=8)

# ── Scale bar ─────────────────────────────────────────────────────────────────
R          = 6371.0
km_per_deg = R * np.cos(np.deg2rad(lat_mean)) * np.pi / 180.0
bar_deg    = 25 / km_per_deg
xlim, ylim = ax.get_xlim(), ax.get_ylim()

sb_x = xlim[1] - 0.15 * (xlim[1] - xlim[0])
sb_y = ylim[1] - 0.03 * (ylim[1] - ylim[0])

ax.plot([sb_x, sb_x + bar_deg], [sb_y, sb_y],
        color='white', lw=2.5, solid_capstyle='butt', zorder=12)
for xpt in [sb_x, sb_x + bar_deg]:
    ax.plot([xpt, xpt], [sb_y - 0.004, sb_y + 0.004],
            color='white', lw=1.5, zorder=12)
ax.text(sb_x,             sb_y - 0.012, '0',
        ha='center', va='top', fontsize=7,
        color='white', fontweight='bold', zorder=12)
ax.text(sb_x + bar_deg,   sb_y - 0.012, '25 km',
        ha='center', va='top', fontsize=7,
        color='white', fontweight='bold', zorder=12)

# ── INSET MAP ────────────────────────────────────────────────────────────────
ax_inset = fig.add_axes([0.65, 0.25, 0.15, 0.15],
                          projection=ccrs.SouthPolarStereo(central_longitude=180))
ax_inset.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())
ax_inset.add_feature(cfeature.LAND,  facecolor='#aaaaaa', zorder=1)
ax_inset.add_feature(cfeature.OCEAN, facecolor='#d0e8f5', zorder=0)
ax_inset.coastlines(resolution='110m', color='white',
                    linewidth=0.5, zorder=2)

study_lons = [LON_BOUNDS[0], LON_BOUNDS[1],
              LON_BOUNDS[1], LON_BOUNDS[0], LON_BOUNDS[0]]
study_lats = [LAT_BOUNDS[0], LAT_BOUNDS[0],
              LAT_BOUNDS[1], LAT_BOUNDS[1], LAT_BOUNDS[0]]
ax_inset.plot(study_lons, study_lats,
              color='tab:red', lw=1.2,
              transform=ccrs.PlateCarree(), zorder=3)
ax_inset.fill(study_lons, study_lats,
              color='tab:red', alpha=0.3,
              transform=ccrs.PlateCarree(), zorder=3)

theta  = np.linspace(0, 2 * np.pi, 100)
center = [0.5, 0.5]
radius = 0.5
verts  = np.vstack([np.sin(theta), np.cos(theta)]).T
circle = mpath.Path(verts * radius + center)
ax_inset.set_boundary(circle, transform=ax_inset.transAxes)

for spine in ax_inset.spines.values():
    spine.set_edgecolor('white')
    spine.set_linewidth(0.7)

plt.subplots_adjust(left=0.1, right=0.95, top=0.95, bottom=0.08)
# plt.savefig('trough_study_site_pipers.png',
#             dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
gsw.SP_from_SA(np.array(34.9),1100,pipers.lon.isel(station=0),pipers.lat.isel(station=0))

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).sal,pipers.isel(station=st).depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.72, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Absolute Salinity [g/kg]')
plt.title('PIPERS CTD Data')


In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).Potential_Temperature,pipers.isel(station=st).depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=-1.85, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Potential Temp')
plt.title('PIPERS CTD Data')

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(palmer.station)):
    plt.plot(palmer.isel(station=st).Absolute_Salinity,palmer.isel(station=st).Depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.9, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Absolute Salinity [g/kg]')
plt.title('Palmer CTD Data')
plt.xlim(33.6,35.25)

In [ ]:
def hssw_thickness_1(ds, salinity_var="SA", depth_var="depth", threshold=34.9):
    """
    Compute HSSW layer thickness for each station in a CTD dataset.

    HSSW top boundary = shallowest depth below which ALL samples
    remain above the salinity threshold (i.e. the layer is contiguous
    down to the bottom of the cast). This is more robust to noisy
    spikes near the boundary than a simple "first exceedance" approach.

    Thickness = max profile depth - HSSW top depth

    Returns a DataArray of thickness values (one per station), with NaN
    for stations that never reach the HSSW threshold, or where the
    bottom-most sample itself is below threshold (no HSSW at depth).
    """
    thicknesses = []
    station_ids = []

    for stn in ds["station"].values:
        profile = ds.sel(station=stn)

        depth = profile[depth_var].values
        salt = profile[salinity_var].values

        # drop NaNs
        valid = np.isfinite(depth) & np.isfinite(salt)
        depth = depth[valid]
        salt = salt[valid]

        if depth.size == 0:
            thicknesses.append(np.nan)
            station_ids.append(stn)
            continue

        # sort by depth ascending (shallow -> deep), in case sample
        # order doesn't already correspond to depth order
        order = np.argsort(depth)
        depth = depth[order]
        salt = salt[order]

        max_depth = depth[-1]

        # working from the bottom (deepest) sample upward, find the
        # shallowest index from which every sample to the bottom is
        # above threshold
        above = salt > threshold

        if not above[-1]:
            # bottom of cast isn't even HSSW -> no HSSW layer at depth
            thickness = np.nan
        else:
            # walk upward from the bottom while still above threshold
            idx = len(above) - 1
            while idx > 0 and above[idx - 1]:
                idx -= 1
            hssw_top_depth = depth[idx]
            thickness = max_depth - hssw_top_depth

        thicknesses.append(thickness)
        station_ids.append(stn)

    return xr.DataArray(
        thicknesses,
        coords={"station": station_ids},
        dims="station",
        name="HSSW_thickness"
    )

In [ ]:
thickness_pipers_1 = hssw_thickness_1(pipers,salinity_var='Absolute_Salinity', depth_var='depth',threshold=34.9)
thickness_palmer_1 = hssw_thickness_1(palmer,salinity_var='Absolute_Salinity',depth_var='Depth',threshold=34.9)

In [ ]:
thickness_palmer_1

In [ ]:
thickness_pipers_1.mean()

In [ ]:
thickness_pipers_1.median()

In [ ]:
thickness_pipers_1.min().values, thickness_pipers_1.max().values

In [ ]:
all_thickness = xr.concat([thickness_pipers_1, thickness_palmer_1], dim="station")

mean_thickness = all_thickness.mean(skipna=True)
n_with_hssw = all_thickness.notnull().sum().item()
n_total = all_thickness.size

print(f"Mean HSSW thickness: {mean_thickness.values:.1f} m")
print(f"HSSW present in {n_with_hssw}/{n_total} profiles")

In [ ]:
all_thickness.max()

In [ ]:
N = np.sqrt(pipers.where(pipers.depth > 500,drop=True).N2.mean()).values

In [ ]:
N

In [ ]:
H = 500

In [ ]:
import gsw

In [ ]:
f = gsw.f(pipers.isel(station=1).lat)

In [ ]:
(N*H/np.abs(f))/1000

In [ ]:
# plot mooring D to see if HSSW always present at 550 meters 
fig = plt.figure(figsize=(12,4))
plt.plot(D_ctd_top.time,D_ctd_top.abs_sal,label='550m')
plt.plot(D_ctd_mid.time,D_ctd_mid.abs_sal, label='770m')
plt.axhline(y=34.9, color='r', linestyle='--', linewidth=2, label='HSSW Threshold')
plt.legend()